In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('ALS_Training') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.driver.memory', '1g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 20:34:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.0


## 1. Load Data

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType

train = spark.read.parquet('/data/train_ratings.parquet')
test  = spark.read.parquet('/data/test_ratings.parquet')

# ALS needs integer columns — use the pre-encoded int columns from load_data.py
def prep(df):
    return df.select(
        F.col('user_id_int').cast(IntegerType()).alias('user_id'),
        F.col('item_id_int').cast(IntegerType()).alias('item_id'),
        F.col('rating').cast(FloatType())
    )

train_df = prep(train)
test_df  = prep(test)

print(f'Train: {train_df.count():,}  |  Test: {test_df.count():,}')
train_df.show(5)

Train: 1,600,000  |  Test: 400,000


+-------+-------+------+
|user_id|item_id|rating|
+-------+-------+------+
|1747837| 138304|   5.0|
|1235934| 414173|   5.0|
|1021694| 265366|   4.0|
| 890360| 144852|   5.0|
|1436158|  66173|   5.0|
+-------+-------+------+
only showing top 5 rows



## 2. Initial ALS Training

In [3]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

def train_als(train_df, rank=10, reg_param=0.1, max_iter=10):
    als = ALS(
        rank=rank,
        regParam=reg_param,
        maxIter=max_iter,
        userCol='user_id',
        itemCol='item_id',
        ratingCol='rating',
        coldStartStrategy='drop',   # drop unknown users/items in test
        nonnegative=True
    )
    return als.fit(train_df)

def evaluate(model, test_df):
    preds = model.transform(test_df)
    evaluator = RegressionEvaluator(
        metricName='rmse',
        labelCol='rating',
        predictionCol='prediction'
    )
    return evaluator.evaluate(preds)

print('Training initial ALS model (rank=10, regParam=0.1, maxIter=10)...')
model = train_als(train_df)
rmse  = evaluate(model, test_df)
print(f'Initial RMSE: {rmse:.4f}')

Training initial ALS model (rank=10, regParam=0.1, maxIter=10)...


[Stage 120:============================>                            (1 + 1) / 2]

Initial RMSE: 2.6512


## 3. Tune if RMSE > 1.5

In [4]:
RMSE_THRESHOLD = 1.5

# Grid of candidates to try when RMSE is too high
TUNING_CONFIGS = [
    {'rank': 20,  'reg_param': 0.05,  'max_iter': 15},
    {'rank': 50,  'reg_param': 0.01,  'max_iter': 20},
    {'rank': 10,  'reg_param': 0.001, 'max_iter': 20},
]

best_model = model
best_rmse  = rmse
best_cfg   = {'rank': 10, 'reg_param': 0.1, 'max_iter': 10}

if rmse > RMSE_THRESHOLD:
    print(f'RMSE {rmse:.4f} > {RMSE_THRESHOLD} — starting tuning...')
    for cfg in TUNING_CONFIGS:
        print(f"  Trying: {cfg}")
        m = train_als(train_df, **cfg)
        r = evaluate(m, test_df)
        print(f"  RMSE: {r:.4f}")
        if r < best_rmse:
            best_rmse  = r
            best_model = m
            best_cfg   = cfg
        if best_rmse <= RMSE_THRESHOLD:
            print('  Threshold met — stopping early.')
            break
    print(f'\nBest config: {best_cfg}  |  Best RMSE: {best_rmse:.4f}')
else:
    print(f'RMSE {rmse:.4f} <= {RMSE_THRESHOLD} — no tuning needed.')

final_model = best_model
final_rmse  = best_rmse
print(f'\nFinal RMSE: {final_rmse:.4f}')

RMSE 2.6512 > 1.5 — starting tuning...
  Trying: {'rank': 20, 'reg_param': 0.05, 'max_iter': 15}


  RMSE: 2.6555
  Trying: {'rank': 50, 'reg_param': 0.01, 'max_iter': 20}


  RMSE: 2.5118
  Trying: {'rank': 10, 'reg_param': 0.001, 'max_iter': 20}


[Stage 656:============================>                            (1 + 1) / 2]

  RMSE: 8.1321

Best config: {'rank': 50, 'reg_param': 0.01, 'max_iter': 20}  |  Best RMSE: 2.5118

Final RMSE: 2.5118


## 4. Save Model

In [5]:
MODEL_PATH = '/data/als_model'

final_model.write().overwrite().save(MODEL_PATH)
print(f'Model saved → {MODEL_PATH}')

Model saved → /data/als_model


## 5. Generate Top-5 Recommendations per User

In [ ]:
RECS_PATH = '/data/user_top5_recs.parquet'

top5_raw = final_model.recommendForAllUsers(5)

top5 = (
    top5_raw
    .withColumn('rec', F.explode('recommendations'))
    .select(
        F.col('user_id'),
        F.col('rec.item_id').alias('item_id'),
        F.col('rec.rating').alias('predicted_rating')
    )
)

top5.write.mode('overwrite').parquet(RECS_PATH)
print(f'Top-5 recs saved → {RECS_PATH}')
print(f'Total rows: {top5.count():,}')
top5.show(10)

[Stage 788:==>                                                    (5 + 1) / 100]

## 6. Summary

In [ ]:
print('=== Training Summary ===')
print(f'Final RMSE          : {final_rmse:.4f}')
print(f'Tuning applied      : {"Yes" if initial_rmse > RMSE_THRESHOLD else "No"}'  if 'initial_rmse' in dir() else f'Tuning applied      : {"Yes" if rmse > RMSE_THRESHOLD else "No"}')
print(f'Best config         : {best_cfg}')
print(f'Model path          : {MODEL_PATH}')
print(f'Recs path           : {RECS_PATH}')

spark.stop()

In [ ]:
final_model.userFactors